## Analysis & figures for the thesis

Turns `results_experiment1_clean.csv` / `results_experiment2.csv` (produced by
`experiment.py`) into every table and figure used in the thesis's Experimental
Results section (Section 5): Tables 1-2, and `figures/fig1_runtime_vs_ratio.png`,
`fig2_success_vs_ratio_n25.png`, `fig4_runtime_by_algorithm.png`,
`fig5_RoundRobin_n12_m42_map_by_objective_{1..4}_value.png`,
`fig6_runtime_vs_ratio_all_algorithms.png`, `fig7_success_vs_ratio_all_algorithms.png`,
and `fig8_objective_severity_vs_ratio_n16.png` (matching `BEP/figures`). Code that
produced earlier, unused draft figures (not present in `BEP/figures`) has been removed.

In [ ]:
# xlsxwriter is used below to write the Table 1 / Table 2 summary workbooks
# (exp1_summary_tables.xlsx, exp2_summary_tables.xlsx) with colour-scale
# formatting; the code falls back to openpyxl automatically if unavailable.
!pip install xlsxwriter

In [ ]:
# Setup: plotting/data-analysis imports and the two experiment result
# files. RESULTS_1 / RESULTS_2 are the per-run CSVs produced by
# experiment.py (one row per (algorithm, instance) run) for Experiment 1
# (Section 4.1, "Good-to-Agent Ratio and Warm Starts") and Experiment 2
# (Section 4.2, "Comparing Algorithms in Practice") respectively; figures
# are written to ./figures, matching BEP/figures in the thesis.
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os

RESULTS_1 = "results_experiment1_clean.csv"
RESULTS_2 = "results_experiment2.csv"
OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)

STYLE = dict(figsize=(10, 3.2), dpi=150)
# Warm-start display order used throughout Experiment 1's tables/figures,
# matching the four SA warm starts of Section 3.5 ("Warm Starts"): Random,
# Welfare, Greedy, Round-Robin.
START_ORDER = ["Random", "Welfare", "Greedy", "RoundRobin"]


# Loads one experiment's result CSV and restores is_efx as a proper bool
# (CSV round-trips it as the string "True"/"False").
def load(path):
    df = pd.read_csv(path)
    df["is_efx"] = df["is_efx"].astype(bool)
    return df

In [ ]:
# Experiment 1 (Section 5.1, "Good-to-Agent Ratio and Warm Starts"):
# per-warm-start success rate & runtime summary (console-printed; not the
# thesis's numbered Table 1 -- see exp1_warm_start_raw_success() far below
# for that one), then fig1/fig2.
def exp1_tables_and_figures(df):
    print("\n" + "=" * 70)
    print("TABLE 1 -- Experiment 1: success rate & runtime by warm start")
    print("=" * 70)
    summary = df.groupby("start_method").agg(
        success_rate=("is_efx", "mean"),
        median_runtime_s=("runtime_wall_seconds", "median"),
        mean_runtime_s=("runtime_wall_seconds", "mean"),
        n_runs=("is_efx", "size"),
    ).reindex(START_ORDER)
    print(summary.round(4).to_string())

    success_by_n = df.groupby("n")["is_efx"].mean()
    n_focus = success_by_n.idxmin()
    print(f"\nSuccess rate by n: {success_by_n.round(4).to_dict()}")
    print(f"-> n={n_focus} is where success rate actually varies; that's what "
          f"fig2 and the detail table below focus on, not all three n values.")

    detail = df[df["n"] == n_focus].pivot_table(
        index="start_method", columns="m_over_n", values="is_efx", aggfunc="mean"
    ).reindex(START_ORDER)
    runtime_detail = df[df["n"] == n_focus].pivot_table(
        index="start_method", columns="m_over_n", values="runtime_wall_seconds", aggfunc="median"
    ).reindex(START_ORDER)

    try:
        import xlsxwriter
        engine = "xlsxwriter"
    except ImportError:
        print("xlsxwriter not installed (pip install xlsxwriter for the color-"
              "scale version) -- falling back to openpyxl for exp1_summary_tables.xlsx.")
        engine = "openpyxl"

    xlsx_path = f"{OUTDIR}/exp1_summary_tables.xlsx"
    with pd.ExcelWriter(xlsx_path, engine=engine) as writer:
        summary.round(4).to_excel(writer, sheet_name="Overview (all n)")
        detail.round(4).to_excel(writer, sheet_name=f"n={n_focus} success rate")
        runtime_detail.round(4).to_excel(writer, sheet_name=f"n={n_focus} median runtime")
        if engine == "xlsxwriter":
            wb = writer.book
            ws = writer.sheets[f"n={n_focus} success rate"]
            ws.conditional_format(1, 1, 10, detail.shape[1],
                {"type": "3_color_scale", "min_color": "#C44E52",
                 "mid_color": "#FFFFBF", "max_color": "#55A868"})
    print(f"Wrote {xlsx_path} (Table 1 + the n={n_focus} detail tables -- "
          f"Table 1 was console-only before, now it's actually in a file)")

    # fig1_runtime_vs_ratio.png (Section 5.1): mean runtime vs. m/n, one
    # panel per tested n in {8, 15, 25}, one line per warm start.
    n_values = sorted(df["n"].unique())
    fig, axes = plt.subplots(1, len(n_values), figsize=(4.2 * len(n_values), 3.4),
                              sharey=True, dpi=150)
    if len(n_values) == 1:
        axes = [axes]
    for ax, n in zip(axes, n_values):
        sub = df[df["n"] == n]
        for method in START_ORDER:
            g = sub[sub["start_method"] == method].groupby("m_over_n")["runtime_wall_seconds"]
            mean = g.mean()
            if mean.empty:
                continue
            ax.plot(mean.index, mean.values, marker="o", ms=3, label=method)
        ax.set_title(f"n = {n}")
        ax.set_xlabel("m / n")
        ax.set_yscale("log")
        if ax is axes[0]:
            ax.set_ylabel("mean runtime (s, log scale)")
    axes[-1].legend(fontsize=8, title="warm start")
    fig.suptitle("Experiment 1: mean runtime vs. item-to-agent ratio, by warm start")
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig1_runtime_vs_ratio.png")
    plt.close(fig)

    # fig2_success_vs_ratio_n{n_focus}.png (Section 5.1): EFX success rate
    # vs. m/n, restricted to n_focus -- every warm start already reaches
    # success rate 1 at the other two n values, so only n_focus (=25) is
    # informative here.
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    sub = df[df["n"] == n_focus]
    for method in START_ORDER:
        g = sub[sub["start_method"] == method].groupby("m_over_n")["is_efx"].mean()
        if g.empty:
            continue
        ax.plot(g.index, g.values, marker="o", ms=4, label=method)
    ax.set_xlabel("m / n")
    ax.set_ylabel("EFX success rate")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8, title="warm start")
    ax.set_title(f"Experiment 1: EFX success rate vs. item-to-agent ratio, n={n_focus}\n")
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig2_success_vs_ratio_n{n_focus}.png")
    plt.close(fig)
    return summary

In [ ]:
# Table 2 (Section 5.2, "Comparing Algorithms in Practice"): success
# rate & runtime for all twelve algorithms, collapsed across the full
# Experiment 2 grid.
def exp2_tables_and_figures(df):
    print("\n" + "=" * 70)
    print("TABLE 2 -- Experiment 2: success rate & runtime by algorithm (all 12)")
    print("=" * 70)
    summary = df.groupby("algorithm_name").agg(
        success_rate=("is_efx", "mean"),
        median_runtime_s=("runtime_wall_seconds", "median"),
        mean_runtime_s=("runtime_wall_seconds", "mean"),
        n_runs=("is_efx", "size"),
    ).sort_values("success_rate")
    print(summary.round(4).to_string())

    order = summary.index.tolist()

    # fig4_runtime_by_algorithm.png: runtime distribution (boxplot) by
    # algorithm, same 12-algorithm order as Table 2 above.
    fig, ax = plt.subplots(**STYLE)
    data = [df[df["algorithm_name"] == a]["runtime_wall_seconds"].values for a in order]
    ax.boxplot(data, tick_labels=order, showfliers=False)
    ax.set_yscale("log")
    ax.set_ylabel("runtime (s, log scale)")
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(order, rotation=45, ha="right")
    ax.set_title("Experiment 2: runtime distribution by algorithm")
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig4_runtime_by_algorithm.png")
    plt.close(fig)

    return summary

In [ ]:
def find_worst_algorithm_size(df2, min_trials=20):
    """Ranks (algorithm, n, m) combinations in Experiment 2 by failure
    rate -- a general diagnostic for finding an interesting slice for
    the map deep-dive of Section 5.2.1, "Failure on the Explicit Map",
    rather than assuming it's Round-Robin or guessing which size.
    """
    g = df2.groupby(["algorithm_name", "n", "m"]).agg(
        trials=("is_efx", "size"),
        failures=("is_efx", lambda s: (~s).sum()),
        failure_rate=("is_efx", lambda s: 1 - s.mean()),
    )
    g = g[g["trials"] >= min_trials].sort_values("failure_rate", ascending=False)
    print("\n" + "=" * 70)
    print("Worst (algorithm, n, m) combinations by failure rate")
    print("=" * 70)
    print(g.head(15).to_string())
    return g

In [ ]:
def plot_algorithm_on_map(df2, algorithm, n, m, color_by="is_efx"):
    """The engine behind fig5_*_map*.png: plots one algorithm's runs at
    one fixed (n, m) on the explicit map of Section 2.3, with the
    CON/IND/WSEP (or SEP, when n == m) corners and the m/n-dependent
    boundary curves drawn in. Called below to produce the four-panel
    figure of Section 5.2.1, "Failure on the Explicit Map".

    color_by: "is_efx" (default) for the binary success/failure split,
    or one of "objective_1_value".."objective_4_value" (Section 3.1)
    to instead show a continuous severity gradient.
    """
    sub = df2[(df2["algorithm_name"] == algorithm) & (df2["n"] == n) & (df2["m"] == m)]
    if sub.empty:
        print(f"No rows for {algorithm} at n={n}, m={m}.")
        return

    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
    if color_by == "is_efx":
        for val, color, label in [(True, "#55A868", "EFX found"), (False, "#C44E52", "EFX missed")]:
            s = sub[sub["is_efx"] == val]
            ax.scatter(s["sigma_2"], s["sigma_1"], s=14, alpha=0.7, color=color, label=label)
        show_legend = True
    else:
        sca = ax.scatter(sub["sigma_2"], sub["sigma_1"], s=18, alpha=0.85,
                          c=sub[color_by], cmap="RdYlGn_r", zorder=3)
        fig.colorbar(sca, ax=ax, label=OBJ_COLS.get(color_by, color_by))
        show_legend = False

    south = np.sqrt(n / m)
    diag = np.linspace(0, np.sqrt(n), 200)
    theta = np.linspace(0, np.pi / 4, 200)
    ax.axhline(south, color="grey", lw=1, ls="--")
    ax.plot(diag, diag, color="grey", lw=1, ls="--")
    ax.plot(np.sqrt(n) * np.sin(theta), np.sqrt(n) * np.cos(theta), color="grey", lw=1, ls="--")

    ell = max(1, m // n)
    corners = [
        (0, np.sqrt(n), "CON"),
        (0, south, "IND"),
        (np.sqrt(1 / ell), np.sqrt(1 / ell), "SEP" if n == m else "WSEP"),
    ]
    for x, y, label in corners:
        ax.scatter([x], [y], marker="s", color="black", zorder=5)
        ax.annotate(label, (x, y), textcoords="offset points", xytext=(6, 0))

    ax.set_xlabel(r"$\sigma_2$")
    ax.set_ylabel(r"$\sigma_1$")
    title_suffix = "" if color_by == "is_efx" else f", colored by {color_by}"
    ax.set_title(f"{algorithm} on the explicit map, {n}x{m}{title_suffix}\n")
    if show_legend:
        ax.legend(fontsize=8)
    fig.tight_layout()
    safe_algo = algorithm.replace(" ", "_")
    suffix = "" if color_by == "is_efx" else f"_by_{color_by}"
    fig.savefig(f"{OUTDIR}/fig5_{safe_algo}_n{n}_m{m}_map{suffix}.png")
    plt.close(fig)

In [ ]:
# Cross-checks plain Greedy's success rate under the two different
# generators: i.i.d. uniform (Experiment 1, recovered from SA-Greedy rows
# with zero SA steps) vs. the resampling model (Experiment 2, Section 2.2)
# -- the same kind of generator-dependence comparison Section 6.2 makes for
# Round-Robin.
def cross_experiment_greedy(df1, df2):
    print("\n" + "=" * 70)
    print("Cross-experiment: plain Greedy, iid_uniform (Exp1) vs. resampling (Exp2)")
    print("=" * 70)
    shared_n = sorted(set(df1["n"].unique()) & set(df2["n"].unique()))
    print(f"n values present in BOTH experiments: {shared_n}")

    g1 = df1[(df1["start_method"] == "Greedy") & (df1["num_iterations"] == 0)]
    if len(g1):
        rate1 = g1.groupby("n")["is_efx"].mean()
        print("\nExp 1 (iid_uniform), plain-Greedy success rate recovered from")
        print("SA-Greedy rows with num_iterations == 0 (i.e. SA took zero steps):")
        print(rate1.round(4).to_string())
    else:
        print("\nNo num_iterations==0 SA-Greedy rows found in this file (toy data "
              "is too small/easy for this to be informative -- rerun on the real CSV).")

    if "Greedy" in df2["algorithm_name"].unique():
        rate2 = df2[df2["algorithm_name"] == "Greedy"].groupby("n")["is_efx"].mean()
        print("\nExp 2 (resampling), plain-Greedy success rate (logged directly):")
        print(rate2.round(4).to_string())

In [ ]:
def build_summary_tables_xlsx(df2, path=f"{OUTDIR}/exp2_summary_tables.xlsx"):
    """Table 2's full (algorithm x size) breakdown (Section 5.2), as one
    .xlsx workbook with runtime (median/mean) and success-rate grids
    per algorithm and size, instead of the single
    collapsed-across-the-grid summary printed above: flattened
    single-row column headers, a colour scale so patterns are visible
    at a glance, and frozen header/index while scrolling across ~32
    size columns.
    """
    runtime_median = df2.pivot_table(index="algorithm_name", columns=["n", "m_over_n"],
                                      values="runtime_wall_seconds", aggfunc="median")
    runtime_mean = df2.pivot_table(index="algorithm_name", columns=["n", "m_over_n"],
                                    values="runtime_wall_seconds", aggfunc="mean")
    success_rate = df2.pivot_table(index="algorithm_name", columns=["n", "m_over_n"],
                                    values="is_efx", aggfunc="mean")

    def flatten_cols(df):
        df = df.copy()
        df.columns = [f"n={n}, m/n={ratio}" for n, ratio in df.columns]
        return df

    overview = pd.DataFrame({
        "median_runtime_s": runtime_median.mean(axis=1),
        "mean_runtime_s": runtime_mean.mean(axis=1),
        "overall_success_rate": success_rate.mean(axis=1),
    }).sort_values("overall_success_rate").reset_index()

    try:
        import xlsxwriter
        engine = "xlsxwriter"
    except ImportError:
        print("xlsxwriter not installed (pip install xlsxwriter for the color-"
              "scale version) -- falling back to openpyxl: same 4 sheets, "
              "same readable headers, just without the color scale / frozen "
              "panes formatting below.")
        engine = "openpyxl"

    with pd.ExcelWriter(path, engine=engine) as writer:
        overview.to_excel(writer, sheet_name="Overview", index=False)
        flatten_cols(runtime_median).to_excel(writer, sheet_name="Runtime (median, s)")
        flatten_cols(runtime_mean).to_excel(writer, sheet_name="Runtime (mean, s)")
        flatten_cols(success_rate).to_excel(writer, sheet_name="Success rate")

        if engine == "xlsxwriter":
            wb = writer.book
            num_fmt = wb.add_format({"num_format": "0.0000"})

            for name, ncols, is_rate in [
                ("Overview", len(overview.columns) - 1, False),
                ("Runtime (median, s)", runtime_median.shape[1], False),
                ("Runtime (mean, s)", runtime_mean.shape[1], False),
                ("Success rate", success_rate.shape[1], True),
            ]:
                ws = writer.sheets[name]
                first_data_col = 1
                last_data_col = ncols
                ws.set_column(0, 0, 22)
                ws.set_column(first_data_col, last_data_col, 13, num_fmt)
                ws.freeze_panes(1, first_data_col)
                ws.conditional_format(
                    1, first_data_col, 1000, last_data_col,
                    {"type": "3_color_scale",
                     "min_color": "#C44E52", "mid_color": "#FFFFBF", "max_color": "#55A868"}
                    if is_rate else
                    {"type": "3_color_scale",
                     "min_color": "#55A868", "mid_color": "#FFFFBF", "max_color": "#C44E52"},
                )

    print(f"Wrote {path} ({len(overview)} algorithms x "
          f"{runtime_median.shape[1]} sizes, 4 sheets)")
    return runtime_median, runtime_mean, success_rate

In [ ]:
# Fixed display order and colour for all twelve Experiment 2 algorithms
# (Section 4.2's algorithm list), reused by every fig5/fig6/fig7/fig8 plot
# below.
ALGO_ORDER_EXP2 = ["SA-Random", "SA-Welfare", "SA-Greedy", "SA-RoundRobin",
                    "Greedy", "RoundRobin", "GA-fitness_prop", "GA-tournament",
                    "ILP-1", "ILP-2", "ILP-3", "ILP-4"]
ALGO_COLORS = dict(zip(ALGO_ORDER_EXP2, plt.cm.tab20(np.linspace(0, 1, len(ALGO_ORDER_EXP2)))))

In [ ]:
# Display names for the four EFX objective functions Obj_1..Obj_4 (Section
# 3.1, "Classification of Algorithms"), reused by the fig5 map colouring
# and the fig8 severity plot below.
OBJ_COLS = {
    "objective_1_value": "Obj_1: Total Violation Count",
    "objective_4_value": "Obj_4: Egalitarian Violation Count",
    "objective_2_value": "Obj_2: Total Envy Magnitude",
    "objective_3_value": "Obj_3: Egalitarian Envy Magnitude",
}

In [ ]:
def objective_relationships(df2):
    """Sanity check + insight into how the four objectives of Section
    3.1, "Classification of Algorithms", relate: Obj_1/Obj_2 sum the
    same per-agent quantities (violation count / envy magnitude) that
    Obj_4/Obj_3 take the max of, so Obj_1 >= Obj_4 and Obj_2 >= Obj_3
    must hold for every row by construction. A violation here means a
    bug, not a finding.
    """
    print("\n" + "=" * 70)
    print("Objective relationships")
    print("=" * 70)
    bad1 = (df2["objective_1_value"] < df2["objective_4_value"]).sum()
    bad2 = (df2["objective_2_value"] < df2["objective_3_value"]).sum()
    print(f"Sanity check (sum >= max must hold for every row):")
    print(f"  objective_1 < objective_4 in {bad1} rows (should be 0)")
    print(f"  objective_2 < objective_3 in {bad2} rows (should be 0)")

    corr = df2[list(OBJ_COLS.keys())].corr()
    print("\nCorrelation between the 4 objectives, all rows:")
    print(corr.round(3).to_string())

In [ ]:
def exp2_objective_severity_vs_ratio(df2, n_focus=16):
    """fig8_objective_severity_vs_ratio_n16.png: among FAILING runs
    only, mean severity per objective (Section 3.1) vs. m/n, for all
    twelve algorithms. All four objectives are exactly 0 iff EFX, so
    averaging over every run would mostly just re-measure success
    rate; conditioning on is_efx == False first instead shows how bad
    the misses are, not just how often they happen. One n at a time
    (all five would be 20 panels); n_focus defaults to the largest
    ratio-grid size since that is typically where failures
    concentrate.
    """
    sub_all = df2[(df2["n"] == n_focus) & (~df2["is_efx"])]
    if sub_all.empty:
        print(f"No failing runs at n={n_focus} -- nothing to plot for severity "
              f"(try a smaller n_focus, or this may just mean everything "
              f"succeeded here).")
        return

    fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=150)
    for ax, (col, title) in zip(axes.flat, OBJ_COLS.items()):
        for algo in ALGO_ORDER_EXP2:
            g = sub_all[sub_all["algorithm_name"] == algo].groupby("m_over_n")[col].mean()
            if g.empty:
                continue
            ax.plot(g.index, g.values, marker="o", ms=3, label=algo, color=ALGO_COLORS[algo])
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("m / n")
        ax.set_ylabel("mean value, among FAILING runs only")
    axes[0, 0].legend(fontsize=6, ncol=2)
    fig.suptitle(f"Experiment 2, n={n_focus}: how bad are the misses?\n"
                 f"(all 4 objectives are 0 iff EFX -- conditioned on failure)")
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig8_objective_severity_vs_ratio_n{n_focus}.png")
    plt.close(fig)

In [ ]:
# Runs every analysis above end-to-end: Experiment 1's tables/figures,
# Experiment 2's tables/figures, the cross-experiment and objective-
# relationship checks, and the Experiment 2 map deep-dive -- writing every
# figure/table used in Sections 5.1/5.2 to ./figures.
df1 = load(RESULTS_1)
df2 = load(RESULTS_2)
exp1_tables_and_figures(df1)
exp2_tables_and_figures(df2)
cross_experiment_greedy(df1, df2)
objective_relationships(df2)
exp2_objective_severity_vs_ratio(df2, n_focus=max(df2["n"]))
build_summary_tables_xlsx(df2)
ranked = find_worst_algorithm_size(df2)
print(f"\nFigures and tables written to ./{OUTDIR}/")

In [ ]:
def exp1_warm_start_raw_success(df1):
    """Table 1 (Section 5.1): for each warm start (Section 3.5, "Warm
    Starts"), what fraction of trials were already EFX before any
    annealing step ran (num_iterations == 0)? This isolates each warm
    start's own construction success rate from whatever Simulated
    Annealing adds on top of it -- for Greedy and Round-Robin
    specifically, this is directly comparable to their own standalone
    success rate, since a SA-Greedy row with num_iterations == 0 is
    exactly what plain Greedy would have returned on its own.
    """
    zero_iter = df1[df1["num_iterations"] == 0]
    bad = zero_iter[zero_iter["status"] != "success"]
    if len(bad):
        print(f"WARNING: {len(bad)} rows have num_iterations==0 but "
              f"status != 'success' -- investigate before trusting this table.")

    raw = df1.groupby("start_method").agg(
        raw_success_rate=("num_iterations", lambda s: (s == 0).mean()),
        n_runs=("num_iterations", "size"),
    ).reindex(START_ORDER)
    print("\n" + "=" * 70)
    print("Raw (pre-annealing) success rate by warm start")
    print("=" * 70)
    print(raw.round(4).to_string())
    return raw


exp1_warm_start_raw_success(df1)

In [ ]:
def exp2_sa_warm_start_diagnostic(df2):
    """Diagnostic beyond Table 2's rounded numbers (Section 5.2): the
    full-precision success rate for each SA warm start (Section 3.5),
    plus a same-instances agreement check -- if two warm starts
    succeed or fail on the exact same trials, not just the same count
    of trials, that is a much stronger signal than matching aggregate
    rates alone.
    """
    sa = df2[df2["algorithm_name"].str.startswith("SA-")]
    print("\n" + "=" * 70)
    print("SA warm start success rates -- full precision")
    print("=" * 70)
    print(sa.groupby("algorithm_name")["is_efx"].agg(["mean", "sum", "count"]).to_string())

    pivoted = sa.pivot_table(index=["n", "m", "instance_seed"],
                              columns="algorithm_name", values="is_efx")
    sa_cols = [c for c in pivoted.columns if c.startswith("SA-")]
    print(f"\nPairwise agreement rate (fraction of trials where both warm "
          f"starts got the SAME is_efx outcome) -- 1.0 across the board "
          f"would mean they're behaving identically, not just similarly:")
    for i, a in enumerate(sa_cols):
        for b in sa_cols[i+1:]:
            both = pivoted[[a, b]].dropna()
            agree = (both[a] == both[b]).mean()
            print(f"  {a} vs {b}: {agree:.4f} ({len(both)} shared trials)")


exp2_sa_warm_start_diagnostic(df2)

In [ ]:
def exp2_ratio_figures_2row(df2):
    """fig6_runtime_vs_ratio_all_algorithms.png and
    fig7_success_vs_ratio_all_algorithms.png (Section 5.2): mean
    runtime and EFX success rate vs. item-to-agent ratio m/n, one
    panel per ratio-grid n, all twelve algorithms -- the 2-row layout
    used in the thesis (a single-row, 5-panel-wide layout was tried
    first and dropped as too compressed to read with 12 algorithms on
    it).
    """
    ratio_grid_n = [4, 6, 8, 12, 16]
    sub_all = df2[df2["n"].isin(ratio_grid_n)]
    ncols = int(np.ceil(len(ratio_grid_n) / 2))

    def _plot(value_col, agg, ylabel, title, filename, logy):
        fig, axes = plt.subplots(2, ncols, figsize=(4.2 * ncols, 7.0), dpi=150, sharey=True)
        axes_flat = axes.flatten()
        for ax, n in zip(axes_flat, ratio_grid_n):
            sub = sub_all[sub_all["n"] == n]
            for algo in ALGO_ORDER_EXP2:
                g = sub[sub["algorithm_name"] == algo].groupby("m_over_n")[value_col]
                g = g.mean() if agg == "mean" else g.median()
                if g.empty:
                    continue
                ax.plot(g.index, g.values, marker="o", ms=3, label=algo, color=ALGO_COLORS[algo])
            ax.set_title(f"n = {n}")
            ax.set_xlabel("m / n")
            if logy:
                ax.set_yscale("log")
        axes_flat[0].set_ylabel(ylabel)
        handles, labels = axes_flat[0].get_legend_handles_labels()
        for ax in axes_flat[len(ratio_grid_n):]:
            ax.axis("off")
        fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8,
                   bbox_to_anchor=(0.5, -0.02))
        fig.suptitle(title)
        fig.tight_layout(rect=[0, 0.06, 1, 1])
        fig.savefig(f"{OUTDIR}/{filename}", bbox_inches="tight")
        plt.close(fig)

    _plot("runtime_wall_seconds", "mean", "mean runtime (s, log scale)",
          "Experiment 2: mean runtime vs. item-to-agent ratio, all 12 algorithms",
          "fig6_runtime_vs_ratio_all_algorithms.png", logy=True)
    _plot("is_efx", "mean", "EFX success rate",
          "Experiment 2: success rate vs. item-to-agent ratio, all 12 algorithms",
          "fig7_success_vs_ratio_all_algorithms.png", logy=False)


exp2_ratio_figures_2row(df2)

In [ ]:
# fig5_RoundRobin_n12_m42_map_by_objective_{1,2,3,4}_value.png: the four
# panels of Section 5.2.1's "Failure on the Explicit Map" figure, one per
# EFX objective (Section 3.1). Round-Robin at n=12, m=42 is the thesis's
# example -- Round-Robin is the one algorithm with a clearly weak region
# (Section 5.2), so this is where the map deep-dive is actually shown.
def objective_severity_maps(df2):
    targets = [
        ("RoundRobin", 12, 42),
    ]
    for algorithm, n, m in targets:
        for obj_col in OBJ_COLS:
            sub = df2[(df2["algorithm_name"] == algorithm) & (df2["n"] == n) & (df2["m"] == m)]
            if sub.empty:
                print(f"  SKIP: no rows for {algorithm} at n={n}, m={m}")
                continue
            plot_algorithm_on_map(df2, algorithm, n, m, color_by=obj_col)


objective_severity_maps(df2)